# MAS Financial Institutions Directory

**Workflow:** drop new `FID_YYYY-MM-DD.xls` files into `mas_data/`, then **Run All**.  
The notebook always compares the **two most recent** files (by date in filename).

In [1]:
import importlib, sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML

sys.path.insert(0, str(Path.cwd()))
import mas_scrapper
importlib.reload(mas_scrapper)
from mas_scrapper import run_analysis, list_files

pd.set_option('display.max_colwidth', 80)

## Files used for this analysis

In [2]:
_probe = run_analysis()

print(f"  Previous extract : {_probe['old_date']}  ({_probe['old_path']})")
print(f"  Current extract  : {_probe['new_date']}  ({_probe['new_path']})")
print()
print(f"  Unique institutions previous week : {_probe['old_count']:,}")
print(f"  Unique institutions current week  : {_probe['new_count']:,}")
print(f"  New entries    : {len(_probe['new_entries']):,}")
print(f"  Removed entries: {len(_probe['removed_entries']):,}")

  Previous extract : 2026-06-10  (/config/workspace/prive-dev/mas_data/FID_2026-06-10.xls)
  Current extract  : 2026-06-22  (/config/workspace/prive-dev/mas_data/FID_2026-06-22.xls)

  Unique institutions previous week : 3,630
  Unique institutions current week  : 3,632
  New entries    : 3
  Removed entries: 1


## Filters
Narrow the results. Press **Apply filters** to refresh charts and tables.

In [3]:
sector_widget = widgets.SelectMultiple(
    options=_probe['all_sectors'],
    value=[],
    description='Sector',
    layout=widgets.Layout(width='320px', height='130px'),
    style={'description_width': '80px'},
)

licence_widget = widgets.SelectMultiple(
    options=_probe['all_licence_types'],
    value=[],
    description='Licence',
    layout=widgets.Layout(width='480px', height='130px'),
    style={'description_width': '80px'},
)

name_widget = widgets.Text(
    placeholder='Filter by name (case-insensitive)',
    description='Name',
    layout=widgets.Layout(width='480px'),
    style={'description_width': '80px'},
)

run_btn = widgets.Button(description='Apply filters', button_style='primary', icon='filter')
reset_btn = widgets.Button(description='Reset', button_style='warning', icon='refresh')
help_text = widgets.HTML('<i style="color:#666">Hold Ctrl / Cmd to multi-select. Leave blank = show all.</i>')

filters = widgets.VBox([
    widgets.HBox([sector_widget, licence_widget]),
    name_widget,
    widgets.HBox([run_btn, reset_btn]),
    help_text,
])

## Week-over-week movements by sector

In [4]:
import html as _html

chart_out = widgets.Output(layout={'overflow':'hidden'})
table_new = widgets.HTML()
table_removed = widgets.HTML()
copy_area = widgets.HTML('')

DISPLAY_COLS = ['Organisation Name', 'Sector', 'Licence Type/Status',
                'Activity/Business Type', 'Address', 'Phone Number', 'Website']

def _render_bar_chart(chart_data):
    sectors = chart_data['Sector'].tolist()
    x = np.arange(len(sectors))
    bar_w = 0.35
    fig, ax = plt.subplots(figsize=(max(7, len(sectors) * 1.5), 4.5))
    bars_new = ax.bar(x - bar_w / 2, chart_data['New'], bar_w, label='New', color='#2ecc71', zorder=3)
    bars_out = ax.bar(x + bar_w / 2, chart_data['Removed'], bar_w, label='Removed', color='#e74c3c', zorder=3)
    ax.set_title('Institution movements by sector (WoW)', fontsize=13, pad=12)
    ax.set_xticks(x)
    ax.set_xticklabels(sectors, fontsize=8, ha='center')
    ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.set_ylabel('Count')
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.5, zorder=0)
    for bar in list(bars_new) + list(bars_out):
        if bar.get_height():
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.05,
                    int(bar.get_height()), ha='center', va='bottom', fontsize=9)
    fig.tight_layout()
    plt.show()

def _style_df(df):
    cols = [c for c in DISPLAY_COLS if c in df.columns]
    return (
        df[cols].style
        .set_properties(**{'text-align': 'left', 'white-space': 'pre-wrap', 'font-size': '12px'})
        .set_table_styles([{'selector': 'th',
                            'props': [('background-color', '#2c3e50'), ('color', 'white'),
                                      ('font-size', '12px'), ('text-align', 'left')]}])
        .hide(axis='index')
        .to_html()
    )

def _v(row, col):
    v = row.get(col, '')
    return '' if pd.isna(v) else str(v).strip()

def _format_email_entries(df, full=True):
    entries = []
    for _, row in df.iterrows():
        name    = _v(row, 'Organisation Name')
        licence = _v(row, 'Licence Type/Status')
        line1   = f"{name} ({licence})" if licence else name
        if not full:
            entries.append(line1)
            continue
        activity = _v(row, 'Activity/Business Type')
        phone    = _v(row, 'Phone Number')
        website  = _v(row, 'Website')
        lines = [line1]
        if activity:
            lines.append(activity)
        contact = '  '.join(x for x in [
            f"\U0001f4de {phone}"   if phone   else '',
            f"\U0001f310 {website}" if website else '',
        ] if x)
        if contact:
            lines.append(contact)
        entries.append('\n'.join(lines))
    return '\n\n'.join(entries)

def _section_header(title, colour, count):
    return (
        '<div style="margin:16px 0 6px 0;">'
        '<h3 style="color:' + colour + '; margin:0;">' + title + ' (' + str(count) + ' shown)</h3>'
        '</div>'
    )

def _build_copy_widget(text):
    escaped = _html.escape(text)
    btn_style = (
        'padding:5px 16px; background:#2980b9; color:#fff; border:none;'
        'border-radius:4px; font-size:12px; font-weight:600; cursor:pointer;'
    )
    onclick = (
        "var p=this.previousElementSibling;"
        "var s=this.nextElementSibling;"
        "navigator.clipboard.writeText(p.textContent).then(function(){"
        "s.textContent='\\u2713 Copied to clipboard';"
        "setTimeout(function(){s.textContent='';},3000);"
        "});"
    )
    return (
        '<div style="margin:8px 0;">'
        '<pre style="display:none">' + escaped + '</pre>'
        '<button onclick="' + onclick + '" style="' + btn_style + '">&#128203; Copy both tables</button>'
        '<span style="color:#27ae60; font-size:12px; margin-left:10px;"></span>'
        '</div>'
    )

def refresh(sectors=None, licence_types=None, name_query=''):
    result = run_analysis(
        sectors=list(sectors) if sectors else None,
        licence_types=list(licence_types) if licence_types else None,
        name_query=name_query,
    )
    cessation = result['new_filtered']['Organisation Name'].str.contains(
        'LODGED NOTICE OF CESSATION', case=False, na=False
    )
    result['new_filtered'] = result['new_filtered'][~cessation]
    n_in  = len(result['new_filtered'])
    n_out = len(result['removed_filtered'])

    header_in = _section_header('New institutions this week', '#27ae60', n_in)
    body_in   = _style_df(result['new_filtered']) if n_in else '<i>None match current filters.</i>'
    table_new.value = header_in + body_in

    header_out = _section_header('Removed institutions this week', '#c0392b', n_out)
    body_out   = _style_df(result['removed_filtered']) if n_out else '<i>None match current filters.</i>'
    table_removed.value = header_out + body_out

    added_block   = _format_email_entries(result['new_filtered'],     full=True)  if n_in  else 'None.'
    removed_block = _format_email_entries(result['removed_filtered'], full=False) if n_out else 'None.'
    copy_text = (
        f"Added ({n_in})\n" + '=' * 60 + '\n' + added_block +
        '\n\n' +
        f"Removed ({n_out})\n" + '=' * 60 + '\n' + removed_block
    )
    copy_area.value = _build_copy_widget(copy_text)

    with chart_out:
        chart_out.clear_output(wait=True)
        _render_bar_chart(result['chart_data'])

run_btn.on_click(lambda _: refresh(
    sectors=sector_widget.value or None,
    licence_types=licence_widget.value or None,
    name_query=name_widget.value.strip(),
))

def on_reset(_):
    sector_widget.value = []
    licence_widget.value = []
    name_widget.value = ''
    refresh()

reset_btn.on_click(on_reset)

refresh()


In [5]:
dashboard = widgets.VBox([
    filters,
    chart_out,
    copy_area,
    table_new,
    table_removed,
])

dashboard
